# 03 Split PPE Input Sources

Create the generated YOLO train/validation/test folders from the new PPE v2 source lanes.

The split supports all four classes:

- `0 = person`
- `1 = helmet`
- `2 = vest`
- `3 = cleaning_coverall`


## Purpose

Notebook 03 turns validated input data into the first generated dataset:

```text
data/generated/splits/train/images/
data/generated/splits/train/labels/
data/generated/splits/val/images/
data/generated/splits/val/labels/
data/generated/splits/test/images/
data/generated/splits/test/labels/
```

Source policy:

- `train`: all valid `open_source` samples plus the training portion of `factory_source`.
- `val`: validation portion of `factory_source` only.
- `test`: all valid `test_source` samples only.

The factory train/validation split is stratified by presence of `helmet`, `vest`, and `cleaning_coverall` so validation remains useful for PPE and role-uniform decisions.


## Setup

Find the `v2_pipeline` root and import the source-aware split helper.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import yaml
from IPython.display import display


# Notebook launch directories vary between Jupyter and VS Code. Walk upward so
# every path below can stay relative to v2_pipeline.
def find_v2_root(start: Path) -> Path:
    """Find the v2_pipeline folder from the current working directory."""
    for candidate in (start, *start.parents):
        if candidate.name == "v2_pipeline" and (candidate / "src").exists():
            return candidate
    raise RuntimeError(
        "Could not find v2_pipeline root. Run this notebook from inside the repository."
    )


V2_ROOT = find_v2_root(Path.cwd()).resolve()
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

from src.dataset.split_dataset import collect_yolo_pairs, split_input_sources  # noqa: E402

print(f"v2 root: {V2_ROOT}")

v2 root: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline


## Load Configuration

Paths and split ratios come from YAML. This notebook does not know or care where the repo lives on disk.


In [2]:
config_dir = V2_ROOT / "configs"
with (config_dir / "dataset_config.yaml").open("r", encoding="utf-8") as file_handle:
    dataset_config = yaml.safe_load(file_handle)
with (config_dir / "class_names.yaml").open("r", encoding="utf-8") as file_handle:
    class_config = yaml.safe_load(file_handle)

input_config = dataset_config["input"]
generated_config = dataset_config["generated"]
reports_config = dataset_config["reports"]
split_config = dataset_config["split"]

open_source_dir = V2_ROOT / input_config["open_source"]
factory_source_dir = V2_ROOT / input_config["factory_source"]
test_source_dir = V2_ROOT / input_config["test_source"]
output_splits_dir = V2_ROOT / generated_config["splits"]
report_dir = V2_ROOT / reports_config["validation"]
report_dir.mkdir(parents=True, exist_ok=True)

train_ratio = float(split_config["train"])
val_ratio = float(split_config["val"])
test_ratio = float(split_config.get("test", 0.0))
random_seed = int(dataset_config.get("random_seed", 42))
class_names = {
    int(class_id): str(class_name)
    for class_id, class_name in class_config["names"].items()
}

print(f"open_source: {open_source_dir}")
print(f"factory_source: {factory_source_dir}")
print(f"test_source: {test_source_dir}")
print(f"output splits: {output_splits_dir}")
print(
    f"factory split ratios -> train={train_ratio:.2f}, val={val_ratio:.2f}, test_source copied separately"
)

open_source: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\data\input\open_source
factory_source: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\data\input\factory_source
test_source: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\data\input\test_source
output splits: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\data\generated\splits
factory split ratios -> train=0.80, val=0.20, test_source copied separately


## Inspect Source Pair Counts

This quick check confirms that the expected source folders have matching image-label pairs before copying anything.


In [3]:
# collect_yolo_pairs checks pairing and parses label class counts. It does not
# copy files. factory_source is required because validation comes only from it.
open_pairs = collect_yolo_pairs(open_source_dir, "open_source", required=False)
factory_pairs = collect_yolo_pairs(factory_source_dir, "factory_source", required=True)
test_pairs = collect_yolo_pairs(test_source_dir, "test_source", required=False)

pair_counts_df = pd.DataFrame(
    [
        {
            "source": "open_source",
            "matched_pairs": len(open_pairs),
            "split_role": "train only",
        },
        {
            "source": "factory_source",
            "matched_pairs": len(factory_pairs),
            "split_role": "train + val",
        },
        {
            "source": "test_source",
            "matched_pairs": len(test_pairs),
            "split_role": "test only",
        },
    ]
)
display(pair_counts_df)

,source,matched_pairs,split_role
0,open_source,180,train only
1,factory_source,520,train + val
2,test_source,42,test only


## Run Source-Aware Split

By default this cell refuses to overwrite an existing generated split. Set `OVERWRITE_EXISTING_SPLITS = True` only when you intentionally want to regenerate `data/generated/splits`.


In [4]:
OVERWRITE_EXISTING_SPLITS = False

# The helper creates data/generated/splits only at this point. It prefixes copied
# filenames with their source lane, for example factory_source__image001.jpg, so
# train can safely combine open-source and factory files without collisions.
copy_report_df, split_summary_df, split_warnings_df = split_input_sources(
    open_source_dir=open_source_dir,
    factory_source_dir=factory_source_dir,
    test_source_dir=test_source_dir,
    output_splits_dir=output_splits_dir,
    train_ratio=train_ratio,
    val_ratio=val_ratio,
    seed=random_seed,
    overwrite=OVERWRITE_EXISTING_SPLITS,
)

print("Split complete.")
display(split_summary_df)

Split complete.


,split,num_images,num_labels,num_objects,num_person,num_helmet,num_vest,num_cleaning_coverall,sources
0,train,596,596,4757,1955,1431,1371,0,"factory_source, open_source"
1,val,104,104,733,293,199,241,0,factory_source
2,test,42,42,165,84,39,42,0,test_source


## Save Split Reports

Keep the report set compact: copy assignments for traceability, one summary table, and one warnings table.


In [5]:
copy_report_path = report_dir / "split_assignments.csv"
summary_path = report_dir / "split_summary.csv"
warnings_path = report_dir / "split_warnings.csv"

copy_report_df.to_csv(copy_report_path, index=False)
split_summary_df.to_csv(summary_path, index=False)
split_warnings_df.to_csv(warnings_path, index=False)

print(f"Saved copy assignments: {copy_report_path}")
print(f"Saved split summary: {summary_path}")
print(f"Saved split warnings: {warnings_path}")

Saved copy assignments: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\validation\split_assignments.csv
Saved split summary: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\validation\split_summary.csv
Saved split warnings: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\validation\split_warnings.csv


## Check Helmet / Vest / Cleaning Coverall Balance

This view focuses on the three PPE/role-uniform classes that must be distributed between factory train and validation. Open-source data is train-only by design, so the most important balance check is factory-derived train versus factory-only validation.


In [6]:
balance_columns = [
    "split",
    "sources",
    "num_helmet",
    "num_vest",
    "num_cleaning_coverall",
]
display(split_summary_df[balance_columns])

factory_only_report = copy_report_df.loc[copy_report_df["source"].eq("factory_source")]
factory_balance_df = (
    factory_only_report.groupby("split", as_index=False)[
        ["num_helmet", "num_vest", "num_cleaning_coverall"]
    ]
    .sum()
    .sort_values("split")
)
print("Factory-only balance:")
display(factory_balance_df)

,split,sources,num_helmet,num_vest,num_cleaning_coverall
0,train,"factory_source, open_source",1431,1371,0
1,val,factory_source,199,241,0
2,test,test_source,39,42,0


Factory-only balance:


,split,num_helmet,num_vest,num_cleaning_coverall
0,train,795,978,0
1,val,199,241,0


## Review Split Warnings

Warnings do not always mean the split failed. For example, if only one factory image contains `cleaning_coverall`, it cannot appear in both train and validation. That is useful information before model training.


In [7]:
if split_warnings_df.empty:
    print("No split warnings found.")
else:
    display(split_warnings_df)

,warning_type,details
0,class_missing_from_split,num_cleaning_coverall is missing from train
1,class_missing_from_split,num_cleaning_coverall is missing from val


## Final Checklist Before Notebook 04

- `data/generated/splits/train` contains open-source plus factory training samples.
- `data/generated/splits/val` contains factory samples only.
- `data/generated/splits/test` contains test-source samples only.
- Helmet, vest, and cleaning-coverall distribution was reviewed for train/validation.
- `reports/validation/split_summary.csv` and `split_warnings.csv` were reviewed.
- No augmentation or model training has run yet.
